# 📊 Distribución Binomial y Aproximación de Poisson con Datos Reales
### Colegio Universitario de Cartago — Big Data
### Probabilidad e Inferencia Matemática para Big Data

**Elaborado por:** Cambronero Zúñiga Carlos · Calvo Solano Sebastián · Rodríguez Zúñiga Isaac  
**Profesor:** David Martínez Salazar  
**Fecha:** Marzo, 2026

---

## 🎯 ¿Qué hace este proyecto?

Se usa el dataset **Telco Customer Churn** (de Kaggle) para aplicar dos distribuciones de probabilidad:

| Distribución | ¿Cuándo se usa? |
|---|---|
| **Binomial** | Cuando hay exactamente 2 resultados posibles (éxito/fracaso) y n ensayos fijos |
| **Poisson** | Aproximación de la Binomial cuando n es grande y p es pequeño |

La **variable de interés** es `Churn`: ¿el cliente canceló el servicio? (Sí = éxito / No = fracaso)

---
## 📦 0. Importar librerías

In [ ]:
# Librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import binom, poisson
from math import comb, factorial, exp

# Estilo visual
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print("✅ Librerías cargadas correctamente")

---
## 📂 1. Descripción del Dataset — Telco Customer Churn

El dataset contiene información de **7,043 clientes** de una empresa de telecomunicaciones.

### Variables principales:
- `CustomerID` → ID único del cliente
- `Gender` → Género
- `SeniorCitizen` → ¿Es adulto mayor?
- `Tenure` → Meses como cliente
- `Churn` → ¿Canceló el servicio? (**nuestra variable binaria**)

### ¿Por qué `Churn` es perfecta para Binomial?
Porque tiene **exactamente 2 resultados**:
- `Yes` = Éxito (canceló) → **p**
- `No` = Fracaso (no canceló) → **q = 1 - p**

In [ ]:
# ─────────────────────────────────────────────
# DATOS EXTRAÍDOS DEL DATASET (Telco Customer Churn)
# ─────────────────────────────────────────────
total_clientes = 7043
churn_si = 1869    # clientes que SÍ cancelaron (éxito)
churn_no = 5174    # clientes que NO cancelaron (fracaso)

# Probabilidades
p = churn_si / total_clientes   # probabilidad de éxito
q = 1 - p                       # probabilidad de fracaso

# Tabla descriptiva
tabla = pd.DataFrame({
    'Categoría': ['Yes (Éxito — Churn)', 'No (Fracaso — No Churn)', 'TOTAL'],
    'Frecuencia Absoluta (fi)': [churn_si, churn_no, total_clientes],
    'Frecuencia Relativa (hi)': [round(p, 4), round(q, 4), 1.0000]
})

print("\n📊 Análisis Descriptivo de la Variable Churn")
print("=" * 55)
print(tabla.to_string(index=False))
print()
print(f"  p (prob. de Churn)    = {churn_si}/{total_clientes} ≈ {p:.4f}")
print(f"  q (prob. de no Churn) = {churn_no}/{total_clientes} ≈ {q:.4f}")
print(f"  Verificación: p + q   = {p + q:.4f} ✅")

In [ ]:
# Gráfico de distribución de Churn
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Gráfico de barras
colores = ['#E74C3C', '#2ECC71']
axes[0].bar(['Churn = Yes', 'Churn = No'], [churn_si, churn_no],
            color=colores, edgecolor='black', linewidth=1.2)
axes[0].set_title('Frecuencia Absoluta de Churn', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Número de Clientes')
for i, v in enumerate([churn_si, churn_no]):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold', fontsize=12)

# Gráfico de torta
axes[1].pie([p, q], labels=[f'Churn=Yes\n({p:.2%})', f'Churn=No\n({q:.2%})'],
            colors=colores, autopct='%1.2f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Distribución Relativa de Churn', fontsize=13, fontweight='bold')

plt.suptitle('Dataset: Telco Customer Churn (n = 7,043)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 🎲 2. Sección B — Probabilidades Exactas con n = 20

### Configuración del modelo

Se seleccionan **20 clientes al azar** del dataset. Sea X la variable aleatoria:

$$X \sim Binomial(n=20, \; p=0.2654)$$

**Fórmula binomial:**

$$P(X = k) = \binom{n}{k} \cdot p^k \cdot q^{n-k}$$

Donde $\binom{n}{k} = \frac{n!}{k!(n-k)!}$ es el **coeficiente binomial** (número de formas de elegir k éxitos en n ensayos).

In [ ]:
# Parámetros del modelo binomial
n = 20
# p y q ya fueron definidos arriba (p ≈ 0.2654)

print(f"Modelo: X ~ Binomial(n={n}, p={p:.4f})")
print(f"q = 1 - p = {q:.4f}")
print()

# ─────────────────────────────────────────────────────────────────
# Función que calcula P(X=k) de forma manual (paso a paso visible)
# ─────────────────────────────────────────────────────────────────
def binomial_manual(n, k, p):
    """Calcula P(X=k) usando la fórmula binomial paso a paso."""
    q = 1 - p
    coef = comb(n, k)           # C(n,k)
    prob = coef * (p**k) * (q**(n-k))
    return coef, p**k, q**(n-k), prob

# Casos a calcular
casos = [0, 1, 2, 3, 10, 15, 20]

print("📋 Probabilidades Exactas P(X = k)")
print("=" * 65)
print(f"{'k':>4} | {'C(20,k)':>10} | {'p^k':>12} | {'q^(20-k)':>12} | {'P(X=k)':>10}")
print("-" * 65)

resultados = {}
for k in casos:
    coef, pk, qnk, prob = binomial_manual(n, k, p)
    resultados[k] = prob
    print(f"{k:>4} | {coef:>10,} | {pk:>12.6f} | {qnk:>12.6f} | {prob:>10.6f}")

print()
print("💡 Nota: P(X=20) es prácticamente 0 porque p=0.2654 es bajo.")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Valor Esperado, Varianza y Desviación Estándar
# ─────────────────────────────────────────────────────────────────
E_X = n * p               # Valor esperado
Var_X = n * p * q         # Varianza
sigma = np.sqrt(Var_X)    # Desviación estándar

print("📐 Medidas de la Distribución Binomial(n=20, p=0.2654)")
print("=" * 50)
print(f"  E(X)  = n·p          = {n} × {p:.4f}         = {E_X:.4f}")
print(f"  Var(X) = n·p·q       = {n} × {p:.4f} × {q:.4f} = {Var_X:.4f}")
print(f"  σ(X)  = √Var(X)      = √{Var_X:.4f}              = {sigma:.4f}")
print()
print(f"  📌 Interpretación: En promedio, de cada 20 clientes,")
print(f"     se esperan ~{E_X:.1f} cancelaciones (churn).")

In [ ]:
# Gráfico de probabilidades exactas para todos los k de 0 a 20
k_vals = np.arange(0, n + 1)
probs = binom.pmf(k_vals, n, p)

# Colores especiales para los k del proyecto
colores_barras = ['#3498DB'] * (n + 1)
for k in casos:
    colores_barras[k] = '#E74C3C'

fig, ax = plt.subplots(figsize=(13, 6))
bars = ax.bar(k_vals, probs, color=colores_barras, edgecolor='black',
              linewidth=0.8, alpha=0.85)

# Etiquetas sobre barras destacadas
for k in casos:
    ax.text(k, probs[k] + 0.003, f'{probs[k]:.4f}', ha='center',
            fontsize=8, fontweight='bold', color='#C0392B')

ax.axvline(E_X, color='#F39C12', linestyle='--', linewidth=2,
           label=f'E(X) = {E_X:.2f}')

ax.set_xlabel('k  (número de clientes con Churn)', fontsize=12)
ax.set_ylabel('P(X = k)', fontsize=12)
ax.set_title(f'Distribución Binomial — n={n}, p={p:.4f}\n'
             f'(Barras rojas = valores analizados en el proyecto)', fontsize=13)
ax.set_xticks(k_vals)

parche_azul = mpatches.Patch(color='#3498DB', alpha=0.85, label='Probabilidad P(X=k)')
parche_rojo = mpatches.Patch(color='#E74C3C', alpha=0.85, label='Valores analizados')
ax.legend(handles=[parche_azul, parche_rojo,
                   plt.Line2D([0],[0], color='#F39C12', lw=2, linestyle='--',
                              label=f'E(X)={E_X:.2f}')])
plt.tight_layout()
plt.show()

---
## 📊 3. Sección C — Probabilidad Acumulada P(X ≤ 5)

$$P(X \leq 5) = P(0) + P(1) + P(2) + P(3) + P(4) + P(5)$$

Se comparan **3 métodos**:
1. **Manual** → Suma de fórmulas individuales con p = 0.2654
2. **Tabla Binomial** → Usando p ≈ 0.25 (valor más cercano en tabla)
3. **Python/scipy** → Cálculo exacto con `binom.cdf()`

In [ ]:
# ─────────────────────────────────────────────────────────────────
# MÉTODO 1: Cálculo Manual — suma de P(k) para k=0..5
# ─────────────────────────────────────────────────────────────────
print("📌 MÉTODO 1 — Cálculo Manual (p = 0.2654)")
print("=" * 50)

suma_manual = 0
for k in range(6):
    _, _, _, prob = binomial_manual(n, k, p)
    suma_manual += prob
    print(f"  P(X={k}) = {prob:.4f}")

print(f"  {'─'*30}")
print(f"  P(X ≤ 5) = {suma_manual:.4f}")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# MÉTODO 2: Tabla Binomial (p ajustado a 0.25)
# ─────────────────────────────────────────────────────────────────
p_tabla = 0.25   # valor más cercano disponible en tablas impresas

prob_tabla = binom.cdf(5, n, p_tabla)

print("📌 MÉTODO 2 — Tabla Binomial (p ajustado a 0.25)")
print("=" * 50)
print(f"  Se ajusta p={p:.4f} → p_tabla={p_tabla}")
print(f"  (Las tablas impresas solo tienen p = 0.05, 0.10, 0.15, 0.20, 0.25, 0.30...)")
print()
print(f"  P(X ≤ 5) con p=0.25 = {prob_tabla:.4f}")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# MÉTODO 3: Python con scipy — EXACTO con p = 0.2654
# binom.cdf(k, n, p) = P(X ≤ k)
# ─────────────────────────────────────────────────────────────────
prob_python = binom.cdf(5, n, p)

print("📌 MÉTODO 3 — Python / scipy (p = 0.2654 exacto)")
print("=" * 50)
print(f"  from scipy.stats import binom")
print(f"  binom.cdf(5, n=20, p={p:.4f})")
print()
print(f"  P(X ≤ 5) = {prob_python:.4f}")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# COMPARACIÓN DE LOS 3 MÉTODOS
# ─────────────────────────────────────────────────────────────────
print("\n📊 COMPARACIÓN — P(X ≤ 5)")
print("=" * 55)
print(f"  {'Método':<30} {'Resultado':>10} {'Diferencia vs Python':>20}")
print(f"  {'─'*53}")
print(f"  {'Manual (p=0.2654)':<30} {suma_manual:>10.4f} {abs(suma_manual-prob_python):>20.4f}")
print(f"  {'Tabla (p=0.25)':<30} {prob_tabla:>10.4f} {abs(prob_tabla-prob_python):>20.4f}")
print(f"  {'Python scipy (p=0.2654)':<30} {prob_python:>10.4f} {'✅ Referencia':>20}")
print()
diff_tabla = abs(prob_tabla - suma_manual)
print(f"  📌 La diferencia Manual vs Tabla es: {diff_tabla:.4f} ({diff_tabla*100:.2f}%)")
print(f"     Esto se debe al ajuste de p: 0.2654 → 0.25")

In [ ]:
# Visualización: probabilidades acumuladas de los 3 métodos
k_rango = np.arange(0, n + 1)
cdf_real   = binom.cdf(k_rango, n, p)
cdf_tabla  = binom.cdf(k_rango, n, p_tabla)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(k_rango, cdf_real,  'b-o', linewidth=2.5, markersize=5,
        label=f'Python/Manual — p={p:.4f}')
ax.plot(k_rango, cdf_tabla, 'r--s', linewidth=2, markersize=5,
        label=f'Tabla — p={p_tabla}')

ax.axvline(5, color='green', linestyle=':', linewidth=2, alpha=0.7, label='k = 5')
ax.annotate(f'P(X≤5)={prob_python:.4f}', xy=(5, prob_python),
            xytext=(7, prob_python - 0.1),
            arrowprops=dict(arrowstyle='->', color='blue'), color='blue', fontsize=11)
ax.annotate(f'P(X≤5)={prob_tabla:.4f}', xy=(5, prob_tabla),
            xytext=(7, prob_tabla + 0.05),
            arrowprops=dict(arrowstyle='->', color='red'), color='red', fontsize=11)

ax.set_xlabel('k', fontsize=12)
ax.set_ylabel('P(X ≤ k)', fontsize=12)
ax.set_title('Función de Distribución Acumulada (CDF) — Binomial\nComparación: p real vs p ajustado para tabla', fontsize=13)
ax.set_xticks(k_rango)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

---
## 📊 4. Sección D — Probabilidad Acumulada P(X ≥ 10)

### ¿Por qué usar el complemento?

Calcular $P(X \geq 10) = P(10) + P(11) + \cdots + P(20)$ directamente requiere 11 cálculos.

**Es mucho más eficiente usar el complemento:**

$$P(X \geq 10) = 1 - P(X \leq 9)$$

In [ ]:
# ─────────────────────────────────────────────────────────────────
# MÉTODO 1: Cálculo Manual — complemento
# ─────────────────────────────────────────────────────────────────
print("📌 MÉTODO 1 — Cálculo Manual con Complemento (p = 0.2654)")
print("=" * 60)

# Ya sabemos P(X ≤ 5) = 0.5544
# Calculamos P(6), P(7), P(8), P(9)
P_leq5 = suma_manual
print(f"  P(X ≤ 5) ya calculado = {P_leq5:.4f}")
print()

P_leq9 = P_leq5
for k in range(6, 10):
    _, _, _, prob = binomial_manual(n, k, p)
    P_leq9 += prob
    print(f"  P(X={k}) = {prob:.4f}")

P_geq10_manual = 1 - P_leq9

print(f"  {'─'*40}")
print(f"  P(X ≤ 9) = {P_leq9:.4f}")
print(f"  P(X ≥ 10) = 1 - {P_leq9:.4f} = {P_geq10_manual:.4f}  ({P_geq10_manual*100:.2f}%)")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# MÉTODO 2: Tabla Binomial (p ajustado a 0.25)
# ─────────────────────────────────────────────────────────────────
P_leq9_tabla = binom.cdf(9, n, p_tabla)
P_geq10_tabla = 1 - P_leq9_tabla

print("📌 MÉTODO 2 — Tabla Binomial (p ajustado a 0.25)")
print("=" * 50)
print(f"  P(X ≤ 9) de tabla = {P_leq9_tabla:.4f}")
print(f"  P(X ≥ 10) = 1 - {P_leq9_tabla:.4f} = {P_geq10_tabla:.4f}  ({P_geq10_tabla*100:.2f}%)")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# MÉTODO 3: Python con scipy — sf() = survival function = 1 - CDF
# binom.sf(k, n, p) = P(X > k) = P(X ≥ k+1)
# Entonces P(X ≥ 10) = binom.sf(9, n, p)
# ─────────────────────────────────────────────────────────────────
P_geq10_python = binom.sf(9, n, p)   # survival function: P(X > 9) = P(X ≥ 10)

print("📌 MÉTODO 3 — Python / scipy (p = 0.2654 exacto)")
print("=" * 50)
print(f"  binom.sf(9, n=20, p={p:.4f})  # sf = 1 - CDF = P(X > 9)")
print(f"  P(X ≥ 10) = {P_geq10_python:.4f}  ({P_geq10_python*100:.2f}%)")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# COMPARACIÓN P(X ≥ 10)
# ─────────────────────────────────────────────────────────────────
print("\n📊 COMPARACIÓN — P(X ≥ 10)")
print("=" * 60)
print(f"  {'Método':<30} {'Resultado':>10} {'Diferencia vs Python':>20}")
print(f"  {'─'*58}")
print(f"  {'Manual (p=0.2654)':<30} {P_geq10_manual:>10.4f} {abs(P_geq10_manual-P_geq10_python):>20.4f}")
print(f"  {'Tabla (p=0.25)':<30} {P_geq10_tabla:>10.4f} {abs(P_geq10_tabla-P_geq10_python):>20.4f}")
print(f"  {'Python scipy (p=0.2654)':<30} {P_geq10_python:>10.4f} {'✅ Referencia':>20}")
print()
diff = abs(P_geq10_manual - P_geq10_tabla)
print(f"  📌 Diferencia Manual vs Tabla: {diff:.4f} ({diff*100:.2f}%)")

In [ ]:
# Visualización: área P(X ≥ 10) resaltada
fig, ax = plt.subplots(figsize=(13, 6))
colores_cdf = ['#2ECC71' if k < 10 else '#E74C3C' for k in k_rango]
probs_exact = binom.pmf(k_rango, n, p)
bars = ax.bar(k_rango, probs_exact, color=colores_cdf,
              edgecolor='black', linewidth=0.8, alpha=0.85)

ax.set_xlabel('k', fontsize=12)
ax.set_ylabel('P(X = k)', fontsize=12)
ax.set_title(
    f'Distribución Binomial (n=20, p={p:.4f})\n'
    f'Verde: P(X < 10) = {1-P_geq10_python:.4f} | Rojo: P(X ≥ 10) = {P_geq10_python:.4f}',
    fontsize=13)
ax.set_xticks(k_rango)

verde = mpatches.Patch(color='#2ECC71', alpha=0.85, label=f'P(X < 10) = {1-P_geq10_python:.4f}')
rojo  = mpatches.Patch(color='#E74C3C', alpha=0.85, label=f'P(X ≥ 10) = {P_geq10_python:.4f}')
ax.legend(handles=[verde, rojo], fontsize=11)
plt.tight_layout()
plt.show()

---
## 🔵 5. Sección E — Aproximación de Poisson a la Distribución Binomial

### ¿Cuándo usar Poisson en lugar de Binomial?

| Condición | Valor en este escenario |
|---|---|
| n grande (≥ 30) | n = 1000 ✅ |
| p pequeño (≤ 0.05) | p = 0.0068 ✅ |
| λ = n·p moderado | λ = 6.8 ✅ |
| Ensayos independientes | ✅ |

### Redefinición del evento
Para cumplir las condiciones, se usa un subgrupo:
- **Evento**: cliente con **2 años de antigüedad** que **hace Churn**
- Total dataset: 7043 | Clientes con 2 años: 1695 | De esos, churn: 48
- **p = 48/7043 ≈ 0.0068** (muy pequeño)
- Se toma muestra de **n = 1000** clientes

$$\lambda = n \cdot p = 1000 \times 0.0068 = 6.8$$

**Fórmula de Poisson:**
$$P(X = k) = \frac{e^{-\lambda} \cdot \lambda^k}{k!}$$

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Parámetros del escenario Poisson
# ─────────────────────────────────────────────────────────────────
n_poisson = 1000
p_poisson = 48 / 7043       # prob. de cliente con 2 años que hace churn
lam = n_poisson * p_poisson  # lambda = parámetro de Poisson

print("⚙️  Parámetros del escenario Poisson")
print("=" * 45)
print(f"  n       = {n_poisson}")
print(f"  p       = 48/7043 = {p_poisson:.6f}")
print(f"  λ = n·p = {n_poisson} × {p_poisson:.4f} = {lam:.4f}")
print()
print("  ✅ Se cumplen todas las condiciones para la aproximación:")
print(f"     n={n_poisson} ≥ 30 ✓ | p={p_poisson:.4f} ≤ 0.05 ✓ | λ={lam:.2f} moderado ✓")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CÓDIGO DEL PROYECTO (código original del docx)
# ─────────────────────────────────────────────────────────────────
from scipy.stats import binom, poisson

n = 1000
p = 0.0068
lam = n * p

# Probabilidad exacta binomial
binom_5 = binom.pmf(5, n, p)
binom_8 = binom.pmf(8, n, p)

# Probabilidad aproximada Poisson
poisson_5 = poisson.pmf(5, lam)
poisson_8 = poisson.pmf(8, lam)

print("k=5 -> Binomial:", round(binom_5, 4), "  Poisson:", round(poisson_5, 4))
print("k=8 -> Binomial:", round(binom_8, 4), "  Poisson:", round(poisson_8, 4))

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Comparación extendida: más valores de k
# ─────────────────────────────────────────────────────────────────
print("\n📊 Comparación extendida: Binomial Exacta vs Aproximación Poisson")
print("=" * 65)
print(f"  {'k':>4} | {'Binomial Exacta':>16} | {'Poisson (aprox)':>16} | {'Diferencia':>12} | {'Calidad':>10}")
print(f"  {'─'*63}")

for k_val in range(0, 16):
    b = binom.pmf(k_val, n, p)
    po = poisson.pmf(k_val, lam)
    diff = abs(b - po)
    calidad = "🟢 Excelente" if diff < 0.001 else ("🟡 Buena" if diff < 0.005 else "🔴 Regular")
    print(f"  {k_val:>4} | {b:>16.6f} | {po:>16.6f} | {diff:>12.6f} | {calidad}")

print()
print("  📌 La aproximación de Poisson es muy precisa en este escenario")
print(f"     porque p={p} es pequeño y n={n} es grande.")

In [ ]:
# Gráfico comparativo: Binomial vs Poisson
k_range = np.arange(0, 20)
binom_probs   = binom.pmf(k_range, n, p)
poisson_probs = poisson.pmf(k_range, lam)

x = np.arange(len(k_range))
width = 0.4

fig, ax = plt.subplots(figsize=(14, 6))
bars1 = ax.bar(x - width/2, binom_probs,  width, label='Binomial Exacta',
               color='#3498DB', edgecolor='black', linewidth=0.7, alpha=0.9)
bars2 = ax.bar(x + width/2, poisson_probs, width, label=f'Poisson (λ={lam:.1f})',
               color='#E74C3C', edgecolor='black', linewidth=0.7, alpha=0.7)

# Marcar k=5 y k=8
for k_mark, b_val, p_val in [(5, binom_5, poisson_5), (8, binom_8, poisson_8)]:
    ax.annotate(f'k={k_mark}\nB={b_val:.3f}\nP={p_val:.3f}',
                xy=(k_mark, max(b_val, p_val)),
                xytext=(k_mark + 1.5, max(b_val, p_val) + 0.01),
                arrowprops=dict(arrowstyle='->', color='black'),
                fontsize=9, ha='left',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.8))

ax.set_xlabel('k  (número de clientes con 2 años que hacen Churn)', fontsize=12)
ax.set_ylabel('Probabilidad', fontsize=12)
ax.set_title(f'Binomial Exacta vs Aproximación de Poisson\n'
             f'n={n}, p={p}, λ={lam:.1f}', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(k_range)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Cálculo manual de Poisson para k=5 y k=8 (verificación de fórmula)
# ─────────────────────────────────────────────────────────────────
lam_val = 6.8

def poisson_manual(lam, k):
    """Calcula P(X=k) con Poisson paso a paso."""
    e_lam = exp(-lam)
    lam_k = lam ** k
    k_fact = factorial(k)
    prob = (e_lam * lam_k) / k_fact
    return e_lam, lam_k, k_fact, prob

print("📐 Verificación de Fórmula de Poisson (λ = 6.8)")
print("=" * 60)

for k_val in [5, 8]:
    e_lam, lam_k, k_fact, prob = poisson_manual(lam_val, k_val)
    print(f"\n  P(X={k_val}):")
    print(f"    = e^(-{lam_val}) × {lam_val}^{k_val} / {k_val}!")
    print(f"    = {e_lam:.6f} × {lam_k:.4f} / {k_fact}")
    print(f"    = {prob:.6f}  ≈  {prob:.3f}")

---
## 📋 6. Resumen General del Proyecto

In [ ]:
# Restaurar p original
p = 1869 / 7043
q = 1 - p
n = 20

print("╔══════════════════════════════════════════════════════════════╗")
print("║          RESUMEN COMPLETO DEL PROYECTO                       ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  Dataset: Telco Customer Churn (n={total_clientes} clientes)           ║")
print(f"║  Variable: Churn  |  p = {p:.4f}  |  q = {q:.4f}            ║")
print("╠══════════════════════════════════════════════════════════════╣")
print("║  SECCIÓN B — Probabilidades Exactas (n=20)                   ║")
print(f"║    E(X) = {n*p:.4f}  |  Var(X) = {n*p*q:.4f}  |  σ = {np.sqrt(n*p*q):.4f}     ║")
print("╠══════════════════════════════════════════════════════════════╣")
print("║  SECCIÓN C — P(X ≤ 5)                                        ║")
print(f"║    Manual (p=0.2654):  {suma_manual:.4f}                            ║")
print(f"║    Tabla  (p=0.25):    {binom.cdf(5,20,0.25):.4f}                            ║")
print(f"║    Python (p=0.2654):  {binom.cdf(5,20,p):.4f}                            ║")
print("╠══════════════════════════════════════════════════════════════╣")
print("║  SECCIÓN D — P(X ≥ 10)                                       ║")
print(f"║    Manual (p=0.2654):  {P_geq10_manual:.4f}                            ║")
print(f"║    Tabla  (p=0.25):    {P_geq10_tabla:.4f}                            ║")
print(f"║    Python (p=0.2654):  {binom.sf(9,20,p):.4f}                            ║")
print("╠══════════════════════════════════════════════════════════════╣")
print("║  SECCIÓN E — Aproximación Poisson                             ║")
print(f"║    n=1000, p=0.0068, λ=6.8                                   ║")
print(f"║    P(X=5): Binomial={binom.pmf(5,1000,0.0068):.4f}  Poisson={poisson.pmf(5,6.8):.4f}   ║")
print(f"║    P(X=8): Binomial={binom.pmf(8,1000,0.0068):.4f}  Poisson={poisson.pmf(8,6.8):.4f}   ║")
print("║    → Diferencia mínima: aproximación EXCELENTE ✅              ║")
print("╚══════════════════════════════════════════════════════════════╝")

---
## 📖 Conclusiones

1. **Distribución Binomial** es el modelo correcto cuando hay exactamente dos resultados, n ensayos fijos e independientes, y una probabilidad constante p. La variable `Churn` cumple perfectamente estas condiciones.

2. **Cálculo Manual vs Python**: Los resultados son prácticamente idénticos. Python es más preciso y eficiente para n grande.

3. **Ajuste por Tabla**: Usar p=0.25 en lugar del valor real p=0.2654 introduce un error de ~6% en P(X≤5) y ~0.7% en P(X≥10). A mayor diferencia entre p real y p de tabla, mayor el error.

4. **Aproximación de Poisson**: Es válida y muy precisa cuando n≥30 y p≤0.05. En este proyecto, con n=1000 y p=0.0068, las diferencias con la binomial exacta son menores a 0.001, confirmando la calidad de la aproximación.

---
## 📚 Bibliografía
- Blastchar (2018). *Telco Customer Churn*. Kaggle. https://www.kaggle.com/datasets/blastchar/telco-customer-churn
- SciPy Documentation: `scipy.stats.binom`, `scipy.stats.poisson`